<a href="https://colab.research.google.com/github/alilouati/data-visualization-using-python/blob/main/Chapter_5/Open_the_Dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Open the Al-Kharj Road Accidents Dashboard
**Dr. Ali Louati · Data Visualization using Python · Chapter 5**

This dashboard joins three skills in one app:
- **Programming:** Python and pandas read 365 days of real accident data (Al-Kharj, 2022).
- **Data mining:** a decision tree and K-Means look for patterns.
- **Visualization:** interactive charts and sliders.

### How to use it
1. Click **Runtime → Run all**.
2. Wait about **1 minute**.
3. The dashboard appears **under Step 2**. You can also click the link to open it in a new tab.

Then try this: open the **🌳 Decision tree** tab and move the **Max tree depth** slider.
Watch the **train** score go up while the **test** score stays close to the simple guess (the baseline).

## Step 1. Get the app and the data (about 30 seconds)

In [ ]:
import os, sys, subprocess, time, urllib.request

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "streamlit"], check=True)

os.makedirs("Chapter_5/dashboard", exist_ok=True)
for path in ["Chapter_5/dashboard/streamlit_app.py", "Chapter_5/2022.csv"]:
    if not os.path.exists(path):
        urllib.request.urlretrieve("https://raw.githubusercontent.com/alilouati/data-visualization-using-python/main/" + path, path)
print("✅ App and data are ready.")

## Step 2. Open the dashboard

In [ ]:
import socket

def port_is_open(port):
    with socket.socket() as s:
        return s.connect_ex(("localhost", port)) == 0

PORT = 8501
if not port_is_open(PORT):
    log = open("streamlit.log", "w")
    subprocess.Popen([sys.executable, "-m", "streamlit", "run", "Chapter_5/dashboard/streamlit_app.py",
                      "--server.port", str(PORT), "--server.headless", "true",
                      "--server.enableCORS", "false", "--server.enableXsrfProtection", "false",
                      "--browser.gatherUsageStats", "false"], stdout=log, stderr=subprocess.STDOUT)
    for _ in range(60):                       # wait until the app answers (max 60 s)
        if port_is_open(PORT):
            break
        time.sleep(1)

if not port_is_open(PORT):
    print("❌ The app did not start. Look at streamlit.log, or use Plan B or Plan C below.")
elif "google.colab" in sys.modules:
    from google.colab import output
    print("✅ The dashboard is running. It is shown below (scroll inside the frame).")
    output.serve_kernel_port_as_window(PORT, anchor_text="👉 Click here to open the dashboard in a new tab")
    output.serve_kernel_port_as_iframe(PORT, height=900)
else:
    print(f"✅ The dashboard is running at http://localhost:{PORT}")

## Plan B. A public link (only if Step 2 shows a blank frame)
Change `USE_PUBLIC_LINK = False` to `True`, then run this cell. It prints a link like `https://....trycloudflare.com` that opens the same dashboard.

In [ ]:
USE_PUBLIC_LINK = False

if USE_PUBLIC_LINK:
    import re
    CF = "/usr/local/bin/cloudflared"
    if not os.path.exists(CF):
        urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", CF)
        os.chmod(CF, 0o755)
    cf = subprocess.Popen([CF, "tunnel", "--url", f"http://localhost:{PORT}", "--no-autoupdate"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for _ in range(80):
        m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", cf.stdout.readline())
        if m:
            print("👉 Open the dashboard here:", m.group(0), "(the first load takes about 10 seconds)")
            break
else:
    print("Plan B is off. Turn it on only if Step 2 shows a blank frame.")

## Plan C. The key results without Streamlit
These two charts always work. They show the same data and the same key lesson as the dashboard.

In [ ]:
import numpy as np, pandas as pd, plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

df = pd.read_csv("Chapter_5/2022.csv", encoding="utf-8-sig")
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True)

# 1) Visualization: accidents inside the city, day by day, with a 7-day average
daily = df[["Date", "CarAccidentsWithinCity"]].assign(**{"7-day average": df.CarAccidentsWithinCity.rolling(7, center=True).mean()})
px.line(daily, x="Date", y=["CarAccidentsWithinCity", "7-day average"],
        title="Car accidents inside Al-Kharj city, 2022").show()

# 2) Data mining: is a day 'busy' (more accidents than usual)? Train vs test score as the tree gets deeper
target = "CarAccidentsWithinCity"
y = np.where(df[target] > df[target].median(), "Busy", "Not busy")
X = df.select_dtypes("number").drop(columns=[target]).assign(weekday=df.Date.dt.dayofweek, month=df.Date.dt.month)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
rows = []
for depth in [1, 2, 3, 4, 5, 6, 8, 10, 12, None]:
    m = DecisionTreeClassifier(max_depth=depth, random_state=42).fit(X_tr, y_tr)
    d = m.get_depth()
    rows += [(d, "Days the tree learned from (train)", m.score(X_tr, y_tr)),
             (d, "New days (test)", m.score(X_te, y_te))]
guess = pd.Series(y_te).value_counts(normalize=True).max()
res = pd.DataFrame(rows, columns=["Tree depth", "Data", "Accuracy"])
fig = px.line(res, x="Tree depth", y="Accuracy", color="Data", markers=True,
              title=f"Train score goes up, test score does not (simple guess = {guess:.1%})")
fig.add_hline(y=guess, line_dash="dash", annotation_text="simple guess")
fig.show()

### What did you learn?
- A deep tree **memorises** the training days, so its train score looks great.
- On **new days** it is only a little better than a simple guess.
- **Always test on new data, and always compare with a simple guess.**